<h1>Data Ingestion and Training By: G5-Rita Group</h1><i>notebook edition</i><br>
1). Obtain<br>
2). Decompress<br>
3). Parse<br>
<li>
<ol>A) Load Game</ol>
<ol>B) Parse Move</ol>
<ol>C) Load Move</ol>
</li>
4). Cleanup

In [1]:
#Define connection to MYSQL Server
ConnectString = "mysql -h database-1.cfvrvxxcdsku.ca-west-1.rds.amazonaws.com -P 3306 -u admin -p --ssl-mode=VERIFY_IDENTITY --ssl-ca=./global-bundle.pem"

host=ConnectString[9:60]
user='admin'
password='Data608-Project'
database='CHESSBOT'


In [2]:
#Define file to load into Database

#Provide two-digit month   (01 -> 12)
File_month = "02"

#Provide four-digit year  (2026  <- 2013 )
File_year = "2018"


In [3]:
### Check if file was loaded already  else load stub and obtain id for latter
import mysql.connector
import files as fs
import time
print(time.ctime()) 
metric = []
successbit = False
CompressedFileName = f"lichess_db_standard_rated_{File_year}-{File_month}.pgn.zst"
Files_id = -1

#connect to SQL server
print("Attempting MySQL Server connection")
connection = mysql.connector.connect(host=host,user=user,password=password,database=database,autocommit=True) 

if connection.is_connected():
    cursor = connection.cursor()
    print("Connected to MySQL Server: ")
    Files_id = fs.files(CompressedFileName,File_month,File_year,cursor)
    if(Files_id == -1):
        print("File: ",CompressedFileName," already loaded")
        STOP()
else:
    print("Failed connection to MySQL Server")
    exit()

Fri Mar 27 00:50:58 2026
Attempting MySQL Server connection
Connected to MySQL Server: 
File Record inserted. ID: 6


In [4]:
####   OBTAIN   the file using bash commands

import subprocess
import os
print(time.ctime()) 

# Define your variables
EBSPath = "/data"
CompressedFileName = f"lichess_db_standard_rated_{File_year}-{File_month}.pgn.zst"
CompressedCompletePath = f"https://database.lichess.org/standard/{CompressedFileName}"

# Ensure the directory exists (wget -P usually creates it, but this is safer)
os.makedirs(EBSPath, exist_ok=True)

# Use subprocess to run the bash wget command
# -P specifies the target directory prefix
try:
    # --show-progress: forces the progress bar to appear
    # --progress=bar:force:noscroll: ensures it stays on one line and doesn't spam
    subprocess.run([
        "wget", 
        "-P", EBSPath, 
        "--show-progress", 
        "--progress=bar:force:noscroll", 
        CompressedCompletePath
    ], check=True)
except subprocess.CalledProcessError as e:
    print(f"Download failed: {e}")

DownloadedCompressedFile = "/data/lichess_db_standard_rated_" + File_year + "-" + File_month + ".pgn.zst"
file_size = os.path.getsize(DownloadedCompressedFile)
print(f"Size: {file_size} bytes")

print(time.ctime()) 

Fri Mar 27 00:50:58 2026


--2026-03-27 00:50:58--  https://database.lichess.org/standard/lichess_db_standard_rated_2018-02.pgn.zst
Resolving database.lichess.org (database.lichess.org)... 141.95.66.62, 2001:41d0:700:5e3e::
Connecting to database.lichess.org (database.lichess.org)|141.95.66.62|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5282843923 (4.9G) [application/octet-stream]
Saving to: ‘/data/lichess_db_standard_rated_2018-02.pgn.zst’

lichess_db_standard  99%[==================> ]   4.91G  11.0MB/s    eta 0s     

Size: 5282843923 bytes
Fri Mar 27 00:56:47 2026


lichess_db_standard 100%[===================>]   4.92G  10.9MB/s    in 5m 48s  

2026-03-27 00:56:47 (14.5 MB/s) - ‘/data/lichess_db_standard_rated_2018-02.pgn.zst’ saved [5282843923/5282843923]



In [5]:
####   DECOMPRESS the file using bash commands
import zstandard as zstd
UnCompressedCompletePath = DownloadedCompressedFile[:-4]
print(time.ctime()) 
dctx = zstd.ZstdDecompressor()

with open(DownloadedCompressedFile, 'rb') as ifh, open(UnCompressedCompletePath, 'wb') as ofh:
    # copy_stream handles the heavy lifting of reading/writing chunks
    dctx.copy_stream(ifh, ofh)
    
print(f"Decompression finished: {UnCompressedCompletePath}")
print(time.ctime()) 

Fri Mar 27 00:56:47 2026
Decompression finished: /data/lichess_db_standard_rated_2018-02.pgn
Fri Mar 27 01:02:06 2026


In [6]:
### PARSE the uncompressed file
print(time.ctime()) 
import parse as pa
metric = []
successbit = False

metric,successbit = pa.Parse(UnCompressedCompletePath,cursor)
print("Total Events Ingested: ",metric[0])
print("Total Events Considered: ",metric[1])
print("Log results: ",metric[2])
print(time.ctime()) 

Fri Mar 27 01:02:06 2026
LinesTotalRead:  100000
LinesTotalRead:  200000
LinesTotalRead:  300000
LinesTotalRead:  400000
LinesTotalRead:  500000
LinesTotalRead:  600000
LinesTotalRead:  700000
LinesTotalRead:  800000
LinesTotalRead:  900000
LinesTotalRead:  1000000
LinesTotalRead:  1100000
LinesTotalRead:  1200000
LinesTotalRead:  1300000
LinesTotalRead:  1400000
LinesTotalRead:  1500000
LinesTotalRead:  1600000
LinesTotalRead:  1700000
LinesTotalRead:  1800000
LinesTotalRead:  1900000
LinesTotalRead:  2000000
LinesTotalRead:  2100000
LinesTotalRead:  2200000
LinesTotalRead:  2300000
LinesTotalRead:  2400000
LinesTotalRead:  2500000
LinesTotalRead:  2600000
LinesTotalRead:  2700000
LinesTotalRead:  2800000
LinesTotalRead:  2900000
LinesTotalRead:  3000000
LinesTotalRead:  3100000
LinesTotalRead:  3200000
LinesTotalRead:  3300000
LinesTotalRead:  3400000
LinesTotalRead:  3500000
LinesTotalRead:  3600000
LinesTotalRead:  3700000
LinesTotalRead:  3800000
LinesTotalRead:  3900000
LinesTota

In [7]:
### Update file details in SQL server

print("Connected to MySQL Server: ")
fs.files_update(Files_id,metric[0],metric[1],file_size,cursor)

print(time.ctime()) 

Connected to MySQL Server: 
Record updated.
Fri Mar 27 01:11:19 2026


In [8]:
### CLEANUP  the drive

import cleanup as cu
print("Starting Cleanup:")
storageRemain = cu.Cleanup("",DownloadedCompressedFile,UnCompressedCompletePath)
print("Output storageRemain: ",storageRemain)
print("Starting Cleanup:")
print(time.ctime()) 

Starting Cleanup:
Output storageRemain:  356673.07421875
Starting Cleanup:
Fri Mar 27 01:11:19 2026
